# 01 — Download and Register the VLM (serverless)

Downloads the Qwen2.5-VL weights from Hugging Face and registers them in Unity Catalog with a `production` alias. Notebook 02 resolves the alias, stages the weights, and serves them with vLLM.

**Compute:** serverless CPU. No GPU is needed here.

Differences from the classic version (`3_Video_Batch_Inference_Ray_Core`):
* **No model load.** The classic notebook loaded the full model into memory just to re-save it (which is why it needed a 672 GB driver). Here the weight files are downloaded directly with `snapshot_download`.
* **Registered as a pyfunc wrapper with the weights as an artifact.** Registration is a governance and versioning step; inference always runs in vLLM. This avoids the `mlflow.transformers` multi-modal validation problems (the dict-form vs path-form gotcha) and the dependency on a specific transformers commit.
* **Default model is `Qwen/Qwen2.5-VL-7B-Instruct`** (~16 GB) for a 1×H100 minimal port. For 32B, set `HF_MODEL_ID` to `Qwen/Qwen2.5-VL-32B-Instruct`, `MODEL_NAME` to `qwen2_5_vl-32b`, and `STAGING_DIR` to a Volume path. The weights are ~66 GB, and logging makes a second copy, so both don't fit on local disk. In notebook 02, set the same `MODEL_NAME` and either `QUANTIZATION=fp8` on 1×H100 or `TENSOR_PARALLEL=2` on a multi-GPU node.

In [0]:
%pip install -q "huggingface_hub>=0.30"
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("CATALOG", "main", label="CATALOG")
dbutils.widgets.text("SCHEMA", "vlm_video", label="SCHEMA")  # same as notebook 00
dbutils.widgets.text("HF_MODEL_ID", "Qwen/Qwen2.5-VL-7B-Instruct", label="HF_MODEL_ID")
dbutils.widgets.text("MODEL_NAME", "qwen2_5_vl-7b", label="MODEL_NAME")
# Lowercase: MLflow alias lookups are case-sensitive and notebook 02 reads 'production'.
dbutils.widgets.text("MODEL_ALIAS", "production", label="MODEL_ALIAS")
# Optional: only needed for gated models. Qwen2.5-VL is public.
dbutils.widgets.text("HF_SECRET_SCOPE", "", label="HF_SECRET_SCOPE")
dbutils.widgets.text("HF_SECRET_KEY", "", label="HF_SECRET_KEY")
# Where to download the weights before logging. Empty = local temp disk (fine for 7B).
# For 32B (~66 GB), use a Volume path such as /Volumes/main/vlm_video/transcribe-video/hf_staging:
# logging copies the weights once more, and both copies do not fit on the ~127 GB local disk.
dbutils.widgets.text("STAGING_DIR", "", label="STAGING_DIR")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
HF_MODEL_ID = dbutils.widgets.get("HF_MODEL_ID")
MODEL_NAME = dbutils.widgets.get("MODEL_NAME")
MODEL_ALIAS = dbutils.widgets.get("MODEL_ALIAS")
HF_SECRET_SCOPE = dbutils.widgets.get("HF_SECRET_SCOPE")
HF_SECRET_KEY = dbutils.widgets.get("HF_SECRET_KEY")
STAGING_DIR = dbutils.widgets.get("STAGING_DIR").strip()

UC_MODEL = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
print(f"{HF_MODEL_ID} -> {UC_MODEL}@{MODEL_ALIAS}")

In [0]:
import os, shutil, tempfile, time
from huggingface_hub import snapshot_download

if HF_SECRET_SCOPE and HF_SECRET_KEY:
    os.environ["HF_TOKEN"] = dbutils.secrets.get(scope=HF_SECRET_SCOPE, key=HF_SECRET_KEY)

# The directory name becomes the artifact name ('weights'), which notebook 02 relies on.
# /local_disk0 is not writable on serverless; the temp dir is on the same local disk.
local_root = os.path.join(STAGING_DIR or tempfile.gettempdir(), "vlm_register")
weights_dir = f"{local_root}/weights"
os.makedirs(weights_dir, exist_ok=True)
free_gb = shutil.disk_usage(local_root).free / 1e9
print(f"Staging in {local_root} (free: {free_gb:.0f} GB)")
if not STAGING_DIR and "32B" in HF_MODEL_ID.upper():
    print("Warning: 32B weights plus the logging copy need ~130 GB; set STAGING_DIR to a Volume path.")

t0 = time.time()
snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=weights_dir,
    # Weights, configs, tokenizer and processor files only; skip READMEs, images, etc.
    allow_patterns=["*.safetensors", "*.json", "*.txt", "*.model", "*.tiktoken", "*.jinja"],
)
size_gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(weights_dir) for f in fs) / 1e9
print(f"Downloaded {HF_MODEL_ID} to {weights_dir}: {size_gb:.1f} GB in {(time.time() - t0) / 60:.1f} min")

In [0]:
import json

# Clean up the hub's local cache metadata so it isn't logged as an artifact.
shutil.rmtree(os.path.join(weights_dir, ".cache"), ignore_errors=True)

config = json.load(open(f"{weights_dir}/config.json"))
index = json.load(open(f"{weights_dir}/model.safetensors.index.json"))
expected_shards = sorted(set(index["weight_map"].values()))
missing = [s for s in expected_shards if not os.path.exists(f"{weights_dir}/{s}")]
required = ["config.json", "preprocessor_config.json", "tokenizer_config.json"]
missing += [f for f in required if not os.path.exists(f"{weights_dir}/{f}")]

print("architectures:", config.get("architectures"))
print(f"{len(expected_shards)} weight shards expected, {len(missing)} files missing")
assert not missing, f"Incomplete download, missing: {missing}"
assert "Qwen2_5_VLForConditionalGeneration" in config.get("architectures", []), "Unexpected model architecture"

In [0]:
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()


class VLMWeights(mlflow.pyfunc.PythonModel):
    """Carries the Hugging Face weights as a governed, versioned artifact.

    Inference does not go through this class: notebook 02 stages artifacts/weights
    and loads it into vLLM directly.
    """

    def load_context(self, context):
        self.weights_dir = context.artifacts["weights"]

    def predict(self, context, model_input, params=None):
        raise NotImplementedError("Serve these weights with vLLM; see notebook 02.")


# Unity Catalog requires a signature. It documents the intended interface.
signature = ModelSignature(
    inputs=Schema([
        ColSpec("string", "text_input"),
        ColSpec("string", "video_path"),
    ]),
    outputs=Schema([ColSpec("string", "generated_text")]),
)

In [0]:
with mlflow.start_run(run_name=f"{MODEL_NAME}-register") as run:
    mlflow.log_params({"hf_model_id": HF_MODEL_ID, "weights_gb": round(size_gb, 1)})
    model_info = mlflow.pyfunc.log_model(
        name="vlm",
        python_model=VLMWeights(),
        artifacts={"weights": weights_dir},
        signature=signature,
        pip_requirements=["mlflow"],  # inference deps live in notebook 02
        registered_model_name=UC_MODEL,
    )

version = model_info.registered_model_version
print(f"Registered {UC_MODEL} version {version}")

In [0]:
# Freshly registered versions can take a moment to become visible in UC; retry instead of a fixed sleep.
for attempt in range(10):
    try:
        client.set_registered_model_alias(name=UC_MODEL, alias=MODEL_ALIAS, version=version)
        break
    except Exception as e:
        print(f"  alias attempt {attempt + 1} failed: {str(e)[:120]}")
        time.sleep(5)

mv = client.get_model_version_by_alias(UC_MODEL, MODEL_ALIAS)
assert str(mv.version) == str(version), f"Alias points to v{mv.version}, expected v{version}"

artifacts = mlflow.artifacts.list_artifacts(f"models:/{UC_MODEL}@{MODEL_ALIAS}")
print(f"{UC_MODEL}@{MODEL_ALIAS} -> version {mv.version} (status {mv.status})")
print("Top-level artifacts:", [a.path for a in artifacts])

# Free the local copy.
shutil.rmtree(local_root, ignore_errors=True)